# Phase 4: RUL (Remaining Useful Life) Regression (시계열 잔여 수명 회귀 예측)

이 단계에서는 소모성 패드의 마모와 기계 성능 저하로 인한 교체 시점을 예측하기 위해, 설비가 완전히 멈추거나 부품을 교체하기 직전까지 남아있는 유효 가동 시간인 **잔여 수명(RUL, Remaining Useful Life)**을 수치적으로 직접 추정하는 회귀 모델링 기법을 배웁니다.

## 🎯 실습 목표
1. **RUL 타겟 라벨링 구현**: CMP 연마 패드의 교체 주기(Cycle)별 점진적 수명 감소 수식 모델링
2. **회귀 학습셋 세팅**: 시간 도메인 및 누적 마모 지표 기반의 변수 설정
3. **회귀 부스팅 모델 학습**: XGBoost 및 Random Forest Regressor 학습
4. **수명 추세 시각화**: True RUL vs Predicted RUL 실시간 감소 곡선 도식화

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import xgboost as xgb

# 시각화 설정
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# Phase 1 전처리 데이터 파일 로드
try:
    df_clean = pd.read_csv('./data/cmp_phase1_cleaned.csv')
    print("데이터 로드 성공!")
except FileNotFoundError:
    print("파일이 없습니다. 01_timeseries_preprocessing.ipynb를 실행해 주세요.")

## 1. RUL (Remaining Useful Life) 라벨링 설계
현업 데이터에서는 부품 교체 이력을 바탕으로 RUL을 역산해야 합니다. 
일정 간격(예: 800초 가공)마다 연마 패드 세척 및 교체가 주기적으로 진행된다고 가정하고 잔여 유효 수명을 Piecewise Linear 구조로 생성합니다.

In [ ]:
df_rul = df_clean.copy()

# 800초 단위로 하나의 주기(Cycle) 설정
cycle_length = 800
df_rul['cycle_index'] = (df_rul['timestamp'] // cycle_length).astype(int)

# 각 주기별로 경과 시간 계산
df_rul['elapsed_time'] = df_rul['timestamp'] % cycle_length

# RUL 정의: (주기 최대 시간 - 경과 시간)
df_rul['RUL'] = cycle_length - df_rul['elapsed_time']

# 예측용 추가 파생 피처 구축 : 누적 마찰에 상응하는 누적 진동과 전류 적분값 생성
df_rul['cumulative_vibration'] = df_rul.groupby('cycle_index')['vibration_kalman'].cumsum()
df_rul['cumulative_current'] = df_rul.groupby('cycle_index')['motor_current'].cumsum()

# 시각화를 통한 RUL 형태 확인
plt.figure(figsize=(12, 4))
plt.plot(df_rul['timestamp'][:2000], df_rul['RUL'][:2000], label='패드 잔여 수명 (RUL)', color='navy')
plt.xlabel('Timestamp (seconds)')
plt.ylabel('RUL')
plt.title('CMP Pad Remaining Useful Life (RUL) Ground Truth Profile')
plt.legend()
plt.show()

## 2. RUL 예측 회귀 모델 훈련
누적 연마량 및 물리 센서 데이터를 기반으로 수명을 정량 추정하는 모델을 훈련합니다.

In [ ]:
features_list = ['spindle_speed', 'pressure', 'slurry_flow', 'motor_current', 
                 'vibration_kalman', 'elapsed_time', 'cumulative_vibration', 'cumulative_current']

X = df_rul[features_list]
y = df_rul['RUL']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# XGBoost Regressor 및 Random Forest Regressor 선언 및 학습
xgb_reg = xgb.XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.08, random_state=42)
xgb_reg.fit(X_train, y_train)

rf_reg = RandomForestRegressor(n_estimators=50, max_depth=8, random_state=42, n_jobs=-1)
rf_reg.fit(X_train, y_train)

print("RUL 회귀 모델 기본 훈련 완료")

## 3. 평가 결과 검증 및 수명 예측 곡선 시각화
MAE와 RMSE 평가 점수를 산출하고, 실제 수명 감소 경향과 모델이 예측한 추세 곡선이 얼마나 일치하는지 비교합니다.

In [ ]:
xgb_preds = xgb_reg.predict(X_test)
rf_preds = rf_reg.predict(X_test)

mae = mean_absolute_error(y_test, xgb_preds)
rmse = np.sqrt(mean_squared_error(y_test, xgb_preds))
r2 = r2_score(y_test, xgb_preds)

print("=== RUL XGBoost Regressor 성능 ===")
print(f"MAE  : {mae:.2f} 초")
print(f"RMSE : {rmse:.2f} 초")
print(f"R2   : {r2:.4f}")

# 실제 시간축 대비 예측값의 트렌드를 보기 위해 테스트 데이터를 정렬하여 가시화
test_df = X_test.copy()
test_df['True_RUL'] = y_test
test_df['Pred_RUL'] = xgb_preds
test_df = test_df.sort_values(by='elapsed_time')

plt.figure(figsize=(15, 5))
plt.scatter(test_df['elapsed_time'], test_df['True_RUL'], color='darkgray', s=3, label='Actual RUL')
plt.plot(test_df['elapsed_time'], test_df['Pred_RUL'], color='firebrick', linewidth=2, label='XGBoost Predicted RUL')
plt.xlabel('Elapsed Time in Current Cycle (seconds)', fontsize=12)
plt.ylabel('RUL', fontsize=12)
plt.title('CMP Pad Remaining Useful Life (RUL) Prediction Trend Comparison', fontsize=14)
plt.legend(fontsize=11)
plt.show()